In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("compas-scores-two-years.csv")

features = [
    "age",
    "priors_count",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "c_charge_degree",
]

X = df[features].copy()
y = df["two_year_recid"]

X["c_charge_degree"] = X["c_charge_degree"].map({"F": 1, "M": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.1%}")

Test accuracy: 69.0%


In [8]:
results = pd.DataFrame({
    "race": df.loc[X_test.index, "race"],
    "sex": df.loc[X_test.index, "sex"],
    "age": df.loc[X_test.index, "age_cat"],
    "actual": y_test,
    "predicted": y_pred,
})

no_new_case = results[results["actual"] == 0]
new_case = results[results["actual"] == 1]

# FPR race
print("False Positive Rate (FPR) by race:")
for group in ["African-American", "Asian", "Caucasian", "Hispanic", "Native American", "Other"]:
    group_rows = no_new_case[no_new_case["race"] == group]

    false_positives = (group_rows["predicted"] == 1).sum()
    total = len(group_rows)
    false_positive_rate = false_positives / total

    print(f"{group} FPR: {false_positives}/{total} = {false_positive_rate:.1%}")

# FPR sex
print("\nFalse Positive Rate (FPR) by sex:")
for group in ["Male", "Female"]:
    group_rows = results[results["sex"] == group]

    false_positives = (group_rows["predicted"] == 1).sum()
    total = len(group_rows)
    false_positive_rate = false_positives / total

    print(f"{group} FPR: {false_positives}/{total} = {false_positive_rate:.1%}")

# FPR age
print("\nFalse Positive Rate (FPR) by age:")
for group in ["Less than 25", "25 - 45", "Greater than 45"]:
    group_rows = results[results["age"] == group]

    false_positives = (group_rows["predicted"] == 1).sum()
    total = len(group_rows)
    false_positive_rate = false_positives / total

    print(f"{group} FPR: {false_positives}/{total} = {false_positive_rate:.1%}")

# FNR race
print("\nFalse Negative Rate (FNR) by race:")
for group in ["African-American", "Asian", "Caucasian", "Hispanic", "Native American", "Other"]:
    group_rows = new_case[new_case["race"] == group]

    false_negatives = (group_rows["predicted"] == 0).sum()
    total = len(group_rows)
    false_negative_rate = false_negatives / total

    print(f"{group} FNR: {false_negatives}/{total} = {false_negative_rate:.1%}")
    
# FNR sex
print("\nFalse Negative Rate (FNR) by sex:")
for group in ["Male", "Female"]:
    group_rows = results[results["sex"] == group]

    false_negatives = (group_rows["predicted"] == 0).sum()
    total = len(group_rows)
    false_negative_rate = false_negatives / total

    print(f"{group} FNR: {false_negatives}/{total} = {false_negative_rate:.1%}")

# FNR age
print("\nFalse Negative Rate (FNR) by age:")
for group in ["Less than 25", "25 - 45", "Greater than 45"]:
    group_rows = results[results["age"] == group]

    false_negatives = (group_rows["predicted"] == 0).sum()
    total = len(group_rows)
    false_negative_rate = false_negatives / total

    print(f"{group} FNR: {false_negatives}/{total} = {false_negative_rate:.1%}")

False Positive Rate (FPR) by race:
African-American FPR: 89/362 = 24.6%
Asian FPR: 0/2 = 0.0%
Caucasian FPR: 35/307 = 11.4%
Hispanic FPR: 14/72 = 19.4%
Native American FPR: 0/3 = 0.0%
Other FPR: 4/47 = 8.5%

False Positive Rate (FPR) by sex:
Male FPR: 428/1152 = 37.2%
Female FPR: 58/291 = 19.9%

False Positive Rate (FPR) by age:
Less than 25 FPR: 144/300 = 48.0%
25 - 45 FPR: 300/829 = 36.2%
Greater than 45 FPR: 42/314 = 13.4%

False Negative Rate (FNR) by race:
African-American FNR: 134/374 = 35.8%
Asian FNR: 1/2 = 50.0%
Caucasian FNR: 113/179 = 63.1%
Hispanic FNR: 32/54 = 59.3%
Native American FNR: 2/2 = 100.0%
Other FNR: 24/39 = 61.5%

False Negative Rate (FNR) by sex:
Male FNR: 724/1152 = 62.8%
Female FNR: 233/291 = 80.1%

False Negative Rate (FNR) by age:
Less than 25 FNR: 156/300 = 52.0%
25 - 45 FNR: 529/829 = 63.8%
Greater than 45 FNR: 272/314 = 86.6%


In [ ]:
#import sqlite3
#compas = sqlite3.connect("compas.db")
#cursor = compas.cursor()
#cursor.execute(""" 
#    CREATE TABLE IF NOT EXISTS compas_data (
#        id INTEGER PRIMARY KEY,
#        sex TEXT NOT NULL CHECK (sex IN ('Male', 'Female')),
#        race TEXT NOT NULL CHECK (race IN ('African-American', 'Asian', 'Caucasian', 'Hispanic', 'Native American', 'Other')),
#        age INTEGER NOT NULL CHECK (age >= 0)
#    ) STRICT;
#""")
#compas.close()